<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/neural/01_mlp_backprop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D1 多層パーセプトロンと誤差逆伝播(notebook)

本文:[`01-mlp-backprop.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/neural/01-mlp-backprop.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

def make_circles(n=400, seed=0):
    # 内側の円(クラス 0)と外側の円(クラス 1)。直線では分けられない
    rng = np.random.default_rng(seed)
    r = np.concatenate([rng.normal(1.0, 0.15, n // 2), rng.normal(2.2, 0.15, n - n // 2)])
    th = rng.uniform(0, 2 * np.pi, n)
    X = np.column_stack([r * np.cos(th), r * np.sin(th)])
    t = np.concatenate([np.zeros(n // 2), np.ones(n - n // 2)])
    return X, t

def sigmoid(a):                                             # S5 で作ったもの
    e = np.exp(-np.abs(a))
    return np.where(a >= 0, 1 / (1 + e), e / (1 + e))

def cross_entropy(y, t):
    eps = 1e-12
    return -np.mean(t * np.log(y + eps) + (1 - t) * np.log(1 - y + eps))

X_train, t_train = make_circles(400, seed=0)
X_test, t_test = make_circles(400, seed=1)
plt.figure(figsize=(4, 4)); plt.scatter(*X_train.T, c=t_train, cmap="coolwarm", s=10); plt.axis("equal"); plt.show()

## 0. ロジスティック回帰では分けられない

S5 のロジスティック回帰で学習してみる。決定境界は直線なので、同心円は分けられない。

In [ ]:
Phi = np.column_stack([np.ones(len(X_train)), X_train])
w = np.zeros(3)
for _ in range(2000):
    w -= 0.5 * Phi.T @ (sigmoid(Phi @ w) - t_train) / len(t_train)
print(f"ロジスティック回帰の学習用の正解率:{np.mean((sigmoid(Phi @ w) > 0.5) == t_train):.3f}(半分は当たるだけ)")

## 1. 順伝播

本文の式 (1) を実装する。重みは He の初期化(本文 §3)で作る関数を用意してある。逆伝播で使うので、途中の値(`X`、`Z1`、`A1`)を `cache` に入れて返す。

**手計算**:本文の確認課題1( $D = 2$ 、 $H = 32$ 、 $N = 400$ の形)を答える。

- 手計算:(ここに書く)

In [ ]:
def init_params(D, H, seed=0):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, np.sqrt(2 / D), (D, H)), "b1": np.zeros(H),
            "W2": rng.normal(0, np.sqrt(2 / H), (H, 1)), "b2": np.zeros(1)}

def forward(X, p):
    raise NotImplementedError("TODO: Z1 = X @ W1 + b1、A1 = ReLU(Z1)、z2 = A1 @ W2 + b2、y = sigmoid(z2)。y は .ravel() で長さ N にする")
    return y, {"X": X, "Z1": Z1, "A1": A1}

In [ ]:
p = init_params(2, 32)
y, cache = forward(X_train, p)
check("y の形は (400,)", y.shape, (400,))
check("隠れ層の形は (400, 32)", cache["A1"].shape, (400, 32))
check("y は 0 と 1 のあいだ", [y.min() > 0, y.max() < 1], [True, True])
# 小さな例を手で確かめる:W1 = I、b1 = (0, -1)、W2 = (1, 1)^T、b2 = 0 で、入力 (2, 3) なら A1 = (2, 2)、z2 = 4
p_small = {"W1": np.eye(2), "b1": np.array([0.0, -1.0]), "W2": np.ones((2, 1)), "b2": np.zeros(1)}
check("小さな例:y = sigmoid(4)", forward(np.array([[2.0, 3.0]]), p_small)[0], [1 / (1 + np.exp(-4))])

## 2. 逆伝播と、勾配の確かめ

本文の式 (2)、(3) を実装する。`grads` は、`p` と同じキーと形を持つ辞書にする。

**実装したら、必ず下の数値微分で確かめる**(本文 §2 の IMPORTANT)。

In [ ]:
def backward(y, t, p, cache):
    N = len(t)
    X, Z1, A1 = cache["X"], cache["Z1"], cache["A1"]
    raise NotImplementedError("TODO: delta2 = (y - t).reshape(-1, 1) / N → W2, b2 の勾配 → delta1 = (delta2 @ W2.T) * (Z1 > 0) → W1, b1 の勾配")
    return {"W1": gW1, "b1": gb1, "W2": gW2, "b2": gb2}

In [ ]:
def numerical_grad(X, t, p, key, h=1e-6):
    g = np.zeros_like(p[key])
    for idx in np.ndindex(p[key].shape):
        orig = p[key][idx]
        p[key][idx] = orig + h; lp = cross_entropy(forward(X, p)[0], t)
        p[key][idx] = orig - h; lm = cross_entropy(forward(X, p)[0], t)
        p[key][idx] = orig
        g[idx] = (lp - lm) / (2 * h)
    return g

p_chk = init_params(2, 5, seed=1)
Xs, ts = X_train[:20], t_train[:20]
ys, cs = forward(Xs, p_chk)
grads = backward(ys, ts, p_chk, cs)
for key in ["W1", "b1", "W2", "b2"]:
    check(f"{key}:逆伝播 = 数値微分(形 {p_chk[key].shape})", grads[key], numerical_grad(Xs, ts, p_chk, key), tol=1e-7)

## 3. 学習

本文 §3 の4つのくり返しを書く。

**予想**:隠れ層 32 の 2 層のネットで、評価用の正解率はどれくらいになるか。

- 予想:(ここに書く)

In [ ]:
def train_mlp(X, t, H=32, lr=0.5, n_iters=2000, seed=0):
    p = init_params(X.shape[1], H, seed)
    history = []
    for i in range(n_iters):
        y, cache = forward(X, p)
        history.append(cross_entropy(y, t))
        raise NotImplementedError("TODO: backward で勾配を求め、p の4つの重みを、それぞれ lr * 勾配 だけ引く")
    return p, np.array(history)

In [ ]:
p, history = train_mlp(X_train, t_train)
acc = lambda p, X, t: np.mean((forward(X, p)[0] > 0.5) == t)
print(f"学習用の正解率 {acc(p, X_train, t_train):.3f}、評価用の正解率 {acc(p, X_test, t_test):.3f}")
check("評価用の正解率は 0.95 以上", acc(p, X_test, t_test) >= 0.95, True)

g1, g2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history); ax[0].set_xlabel("iteration"); ax[0].set_ylabel("cross entropy")
ax[1].contourf(g1, g2, forward(grid, p)[0].reshape(g1.shape), levels=20, cmap="coolwarm", alpha=0.6)
ax[1].scatter(*X_train.T, c=t_train, cmap="coolwarm", s=8, edgecolors="k", linewidths=0.2); ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

## 4. 学習された基底関数を見る

2層目はロジスティック回帰で、その入力は隠れ層の出力 $A_1$ である(本文 §1)。隠れ層の素子のいくつかについて、入力の平面上で出力 $\mathrm{ReLU}(x^\top w_{1,j} + b_{1,j})$ を描く。どれも「ある直線の片側で 0、反対側で傾いた平面」になっている。これらを重みをつけて足すと、円に近い境界が作られる。

In [ ]:
_, cache_g = forward(grid, p)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
top = np.argsort(-np.abs(p["W2"].ravel()))[:4]              # 2層目の重みの絶対値が大きい素子を4つ
for ax, j in zip(axes, top):
    ax.contourf(g1, g2, cache_g["A1"][:, j].reshape(g1.shape), levels=20, cmap="viridis")
    ax.set_title(f"unit {j}, W2 = {p['W2'][j, 0]:+.2f}", fontsize=9); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## 5. 条件を変える

1. 隠れ層の幅 $H$ を 1、2、4、32 に変えると、評価用の正解率はどうなるか(本文 §4)
2. 学習率を 0.05、0.5、5 に変えると、損失の曲線はどうなるか
3. 重みをすべて 0 から始めると、どうなるか(本文の確認課題5)

In [ ]:
for H in [1, 2, 4, 32]:
    pH, _ = train_mlp(X_train, t_train, H=H)
    print(f"1. H = {H:2d}:評価用の正解率 {acc(pH, X_test, t_test):.3f}")
for lr in [0.05, 0.5, 5.0]:
    _, hist = train_mlp(X_train, t_train, lr=lr)
    print(f"2. lr = {lr}:損失 100回目 {hist[99]:.3f}、2000回目 {hist[-1]:.3f}")
p0 = {k: np.zeros_like(v) for k, v in init_params(2, 32).items()}
for i in range(2000):
    y0, c0 = forward(X_train, p0); g0 = backward(y0, t_train, p0, c0)
    p0 = {k: p0[k] - 0.5 * g0[k] for k in p0}
print(f"3. 0 から始める:評価用の正解率 {acc(p0, X_test, t_test):.3f}、W1 の値の種類 {len(np.unique(p0['W1'].round(12)))} 個")

## 6. 何層でも使える形にする

R10(DQN)では、層の数を変えられるように、重みを `(W, b)` のリストで持つ形を使う。本文 §2 のパターン(重みの勾配 = 入力$^\top$ × δ、前の層の δ = δ × 重み$^\top$ × 活性化の微分)を、そのままくり返すだけである。この notebook のネットと同じ重みで、同じ勾配になることを確かめる。

In [ ]:
def forward_layers(params, X):
    hs, h = [X], X
    for i, (W, b) in enumerate(params):
        z = h @ W + b
        h = np.maximum(z, 0) if i < len(params) - 1 else z   # 最後の層は活性化なし
        hs.append(h)
    return h, hs

def backward_layers(params, hs, dout):
    grads, d = [], dout
    for i in reversed(range(len(params))):
        W, b = params[i]
        grads.append((hs[i].T @ d, d.sum(axis=0)))
        if i > 0:
            d = (d @ W.T) * (hs[i] > 0)
    return grads[::-1]

layers = [(p_chk["W1"], p_chk["b1"]), (p_chk["W2"], p_chk["b2"])]
out, hs = forward_layers(layers, Xs)
y_l = sigmoid(out).ravel()
g_l = backward_layers(layers, hs, (y_l - ts).reshape(-1, 1) / len(ts))   # 出力層の δ は式 (2)
check("リストの形でも同じ出力", y_l, ys)
check("リストの形でも同じ W1 の勾配", g_l[0][0], grads["W1"])

## 7. 説明(自分の言葉で書く)

1. 隠れ層は何をしているか。「基底関数」という言葉を使って説明する
   - (ここに書く)
2. 誤差逆伝播で、出力層から入力層へ向かって δ を計算するのはなぜか
   - (ここに書く)
3. 勾配を数値微分で確かめることが大切なのはなぜか
   - (ここに書く)